# Medumba ↔ French LoRA Training (Colab)

This notebook trains a **single bidirectional translation adapter** (FR→Medumba and Medumba→FR) using your prepared `pairs_*.jsonl` files.

## What this notebook does
1. Install dependencies
2. Load your dataset
3. Fine-tune `google/byt5-small` with LoRA
4. Run quick evaluation + demo prompts (both directions)
5. Export adapter artifacts for product integration

In [ ]:
import os, json, platform, torch
print('Python:', platform.python_version())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install -U transformers datasets peft accelerate sacrebleu sentencepiece safetensors

## Data input
Choose one option:
- **Option A**: Upload `pairs_train.jsonl`, `pairs_val.jsonl`, `pairs_test.jsonl` manually
- **Option B**: Mount Google Drive and point to your files

In [ ]:
# Option A: manual upload
from google.colab import files
uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

In [ ]:
# Option B: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_DIR = '/content/drive/MyDrive/MedumbaAI/outputs'
# TRAIN_FILE = f'{DATA_DIR}/pairs_train.jsonl'
# VAL_FILE = f'{DATA_DIR}/pairs_val.jsonl'
# TEST_FILE = f'{DATA_DIR}/pairs_test.jsonl'

# Default paths if you uploaded files into /content
TRAIN_FILE = '/content/pairs_train.jsonl'
VAL_FILE = '/content/pairs_val.jsonl'
TEST_FILE = '/content/pairs_test.jsonl'

for p in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    print(p, '=>', os.path.exists(p))

In [ ]:
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq, TrainerCallback
)
import math

print('NOTE_UPDATED: stable training cell v2 loaded')

BASE_MODEL = 'google/byt5-small'
OUTPUT_DIR = '/content/medumba_lora_byt5_bidirectional'
EPOCHS = 20  # start with 10-20; only use 50 if validation improves
BATCH_SIZE = 8
GRAD_ACCUM = 2
LR = 5e-5
MAX_SOURCE_LEN = 128
MAX_TARGET_LEN = 128

def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def add_direction_prefix(row):
    task = row.get('task', '')
    if task == 'fr_to_medumba':
        row['model_input'] = 'fr2med: ' + row['input']
    elif task == 'medumba_to_fr':
        row['model_input'] = 'med2fr: ' + row['input']
    else:
        row['model_input'] = row['input']
    row['model_output'] = row['output']
    return row

train_rows = [add_direction_prefix(r) for r in read_jsonl(TRAIN_FILE)]
val_rows = [add_direction_prefix(r) for r in read_jsonl(VAL_FILE)]

train_ds = Dataset.from_list(train_rows)
val_ds = Dataset.from_list(val_rows)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, use_safetensors=True)

peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=['q', 'v']
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

def tokenize_batch(batch):
    x = tokenizer(batch['model_input'], truncation=True, max_length=MAX_SOURCE_LEN)
    y = tokenizer(text_target=batch['model_output'], truncation=True, max_length=MAX_TARGET_LEN)
    x['labels'] = y['input_ids']
    return x

tokenized_train = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
tokenized_val = val_ds.map(tokenize_batch, batched=True, remove_columns=val_ds.column_names)

# Sanity check: ensure labels are not empty/pad-only
pad_id = tokenizer.pad_token_id
def valid_label_ratio(ds):
    valid = 0
    for row in ds:
        labels = row['labels']
        if any((tok != pad_id) for tok in labels):
            valid += 1
    return valid / len(ds) if len(ds) else 0.0

print('Train valid-label ratio:', valid_label_ratio(tokenized_train))
print('Val valid-label ratio:', valid_label_ratio(tokenized_val))
print('Train rows:', len(tokenized_train), 'Val rows:', len(tokenized_val))

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, label_pad_token_id=-100)

# Stability-first defaults for Colab
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = False

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_ratio=0.03,
    lr_scheduler_type='cosine',
    max_grad_norm=1.0,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    predict_with_generate=False,
    logging_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=use_fp16,
    bf16=use_bf16,
    report_to='none',
    remove_unused_columns=False
)

class StopOnNaNCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return control
        for key in ['loss', 'eval_loss']:
            value = logs.get(key)
            if value is not None and (isinstance(value, float) and (math.isnan(value) or math.isinf(value))):
                print(f'Stopping because {key} became {value}')
                control.should_training_stop = True
                break
        return control

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    callbacks=[StopOnNaNCallback()]
)

trainer.train()
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved adapter + tokenizer to', OUTPUT_DIR)

In [ ]:
# Quick evaluation on test set
from peft import PeftModel
import sacrebleu

test_rows = [add_direction_prefix(r) for r in read_jsonl(TEST_FILE)]

base = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL, use_safetensors=True)
model_eval = PeftModel.from_pretrained(base, OUTPUT_DIR)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_eval.to(device)
model_eval.eval()

def generate(text):
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN)
    enc = {k:v.to(device) for k,v in enc.items()}
    with torch.no_grad():
        out = model_eval.generate(**enc, max_new_tokens=64, num_beams=4, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

subset = test_rows[:150]
preds = []
refs = []
em = 0
for r in subset:
    p = generate(r['model_input'])
    preds.append(p)
    refs.append(r['model_output'])
    em += int(p.strip() == r['model_output'].strip())

chrf = sacrebleu.corpus_chrf(preds, [refs]).score
print({'rows': len(subset), 'exact_match': em/len(subset), 'chrf': chrf})

In [ ]:
# Demo prompts (both directions)
demo_prompts = [
    'fr2med: Bonjour mon ami',
    'fr2med: Je viens bientôt',
    'fr2med: Ma mère vient darriver',
    'med2fr: Ndàʼndàʼa sê’',
    'med2fr: Mə̂ α yǒg ne sə̂’ ə'
]
for t in demo_prompts:
    print('INPUT :', t)
    print('OUTPUT:', generate(t))
    print('-' * 60)

In [ ]:
# Export artifacts
import shutil, os
ARCHIVE_PATH = '/content/medumba_lora_byt5_bidirectional.zip'
if os.path.exists(ARCHIVE_PATH):
    os.remove(ARCHIVE_PATH)
shutil.make_archive('/content/medumba_lora_byt5_bidirectional', 'zip', OUTPUT_DIR)
print('Archive:', ARCHIVE_PATH)

# Optional: download to local machine
from google.colab import files
files.download(ARCHIVE_PATH)

## Product integration notes
- For product, you typically export: `adapter_model.safetensors`, `adapter_config.json`, tokenizer files.
- At runtime, load base model + adapter, then prefix input with `fr2med:` or `med2fr:`.
- If you want a single portable full model, you can merge adapter into base weights (larger artifact).